In [1]:
#test_cloud_cover_local_Google.ipynb
#This one from Google AI instead of Grok test_cloud_cover_local.ipynb (they are fairly similar...)
import ee
import geemap

# Initialize the Earth Engine map
m = geemap.Map()

Enter verification code:  4/1AXlqoi7BWZI9TUegsSu28SGbc-RLR7e-gwxdKSbE9oXUs08qBxjwbbweiRc



Successfully saved authorization token.


In [ ]:
# 1. Define your region of interest (ROI) geometry
# Replace this with your specific geometry, feature, or shapefile
roi = ee.Geometry.Polygon([
    [[-122.5, 37.7], [-122.5, 37.8], [-122.4, 37.8], [-122.4, 37.7]]
])
roi=ee.Geometry.Polygon( #Margerie
    [[[-137.0999444802902, 59.025089518606336],
      [-137.04128095414362, 59.02602709594012],
      [-137.04305404954465, 59.055910019125456],
      [-137.10176844180705, 59.05497133848375],
      [-137.0999444802902, 59.025089518606336]]])

# 2. Load a Landsat 8 or 9 Collection 2 Image
# (Ensure you use the unclipped image here; we reduce over the roi geometry) #example from SF
image = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(roi) \
    .filterDate('2023-01-01', '2023-12-31') \
    .first()

image = ee.Image('LANDSAT/LC08/C02/T1_L2/LC08_059019_20140903') #Mostly clear Margerie, but scene has a lot of cloud

In [12]:
def apply_scale_factors(image):
  opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

image2 = apply_scale_factors(image)

In [16]:
vis_params = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 30000}
vis_params2 = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.6}
m.addLayer(image, vis_params, 'Landsat TOA')
m.addLayer(image2, vis_params2, 'Landsat TOA2')
m

Map(bottom=1240375.0, center=[59.04006891947098, -137.07010681138647], controls=(WidgetControl(options=['posit…

In [4]:
# Clip the image for visualization purposes
clipped_image = image.clip(roi)

# 3. Extract the QA_PIXEL band to determine clouds
qa = image.select('QA_PIXEL')

# Landsat C2 QA_PIXEL Bitmasks:
# Bit 3 = Cloud, Bit 4 = Cloud Shadow
cloud_bit = 1 << 3
shadow_bit = 1 << 4

# Create a binary mask where 1 represents cloud or cloud shadow
cloud_mask = qa.bitwiseAnd(cloud_bit).neq(0).Or(qa.bitwiseAnd(shadow_bit).neq(0))

# 4. Calculate areas using pixelArea() to handle projection distortion
# Compute the total pixel area within the ROI
total_area_img = ee.Image.pixelArea().updateMask(qa.mask())
total_stats = total_area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=30,  # Landsat resolution is 30 meters
    maxPixels=1e10
)
total_area = ee.Number(total_stats.get('area'))

# Compute the cloud-covered pixel area within the ROI
cloud_area_img = ee.Image.pixelArea().updateMask(cloud_mask)
cloud_stats = cloud_area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=30,
    maxPixels=1e10
)
cloud_area = ee.Number(cloud_stats.get('area'))

# 5. Compute the final percentage
# Handle potential division by zero if the ROI is outside the image bounds
cloud_percentage = ee.Algorithms.If(
    total_area.gt(0),
    cloud_area.divide(total_area).multiply(100),
    0
)

# Fetch the result from GEE servers to Python local environment
cloud_pct_value = cloud_percentage.getInfo()
print(f"Cloud coverage for the clipped ROI: {cloud_pct_value:.2f}%")

Cloud coverage for the clipped ROI: 12.68%


In [5]:
cloud_cover_wholeimage=image.get('CLOUD_COVER').getInfo()
cloud_cover_landimage=image.get('CLOUD_COVER_LAND').getInfo()
#One-liner
cloud_cover1 = image.clip(roi).select('QA_PIXEL') \
    .bitwiseAnd(1<<1).Or(ee.Image().byte().paint(roi, 1).bitwiseAnd(1<<3)) \
    .reduceRegion(ee.Reducer.mean(), roi, 30).getInfo()
print(f"Whole image: {cloud_cover_wholeimage}%\nLand whole image {cloud_cover_landimage}%\nCloud + Shadow Cover over AOI: {cloud_pct_value:.2f}% or 1-liner: {cloud_cover1['QA_PIXEL']:.2f}")

Whole image: 81.32%
Land whole image 82.83%
Cloud + Shadow Cover over AOI: 12.68% or 1-liner: 0.08
